# Домашнее задание № 3. Языковые модели

## Задание 1 (8 баллов).

В семинаре для генерации мы использовали предположение маркова и считали, что слово зависит только от 1 предыдущего слова. Но ничто нам не мешает попробовать увеличить размер окна и учитывать два или даже три прошлых слова. Для них мы еще сможем собрать достаточно статистик и, логично предположить, что качество сгенерированного текста должно вырасти.

Попробуйте сделать языковую модель, которая будет учитывать два предыдущих слова при генерации текста.
Сгенерируйте несколько текстов (3-5) и расчитайте перплексию получившейся модели. 
Можно использовать данные из семинара или любые другие (можно брать только часть текста, если считается слишком долго). Перплексию рассчитывайте на 10-50 отложенных предложениях (они не должны использоваться при сборе статистик).


Подсказки:  
    - нужно будет добавить еще один тэг \<start>  
    - можете использовать тот же подход с матрицей вероятностей, но по строкам хронить биграмы, а по колонкам униграммы 
    - тексты должны быть очень похожи на нормальные (если у вас получается рандомная каша, вы что-то делаете не так)
    - у вас будут словари с индексами биграммов и униграммов, не перепутайте их при переводе индекса в слово - словарь биграммов будет больше словаря униграммов и все индексы из униграммного словаря будут формально подходить для словаря биграммов (не будет ошибки при id2bigram[unigram_id]), но маппинг при этом будет совершенно неправильным 

In [149]:
from collections import Counter
from string import punctuation
from razdel import tokenize as razdel_tokenize
from razdel import sentenize
from nltk.tokenize import sent_tokenize
import numpy as np
from scipy.sparse import lil_matrix, csc_matrix
import random

In [150]:
import re

In [151]:

def normalize(text):
    tokens = [w.text for w in razdel_tokenize(text)]    
    normalized = []
    for token in tokens:
        cleaned = token.strip(punctuation + '«»„“""\'\'')
        if not cleaned or len(cleaned) > 20:
            continue
        cleaned = cleaned.lower()
        
        if re.search(r'\d', cleaned):
            if re.match(r'^[\d,. ]+$', cleaned):
                normalized.append('<num>')
            else:
                normalized.append('<num_mixed>')
        else:
            normalized.append(cleaned)
    return normalized

In [152]:
def ngrammer(tokens, n=2):
    ngrams = []
    for i in range(0, len(tokens) - n + 1):
        ngrams.append(' '.join(tokens[i:i+n]))
    return ngrams

In [153]:
corpus_text = open('/Users/kseniazavyalova/Downloads/lenta.txt').read()

In [154]:
len(corpus_text)

11536552

In [155]:
raw_sentences = sent_tokenize(corpus_text)

In [156]:
sentences = []
for text in raw_sentences:
    toks = normalize(text)
    sent = ['<start>', '<start>'] + toks + ['<end>']
    if len(sent) > 3:
        sentences.append(sent)

In [157]:
random.seed(42)
random.shuffle(sentences)

heldout_n = 50
heldout = sentences[:heldout_n]
train_sentences = sentences[heldout_n:]
print(f"Training on {len(train_sentences)} sentences; heldout: {len(heldout)}")


Training on 76293 sentences; heldout: 50


In [158]:
unigrams = Counter()
bigrams = Counter()
trigrams = Counter()

for sent in train_sentences:
    unigrams.update(sent)
    bigrams.update(ngrammer(sent, n=2))
    trigrams.update(ngrammer(sent, n=3))

In [170]:
len(unigrams)

112339

In [171]:
len(bigrams)

746172

In [159]:
id2word = list(unigrams.keys())
word2id = {w:i for i,w in enumerate(id2word)}

id2bigram = list(bigrams.keys())
bigram2id = {b:i for i,b in enumerate(id2bigram)}

V = len(id2word)
matrix_counts = lil_matrix((len(id2bigram), V), dtype=float)

for trigram, cnt in trigrams.items():
    w1, w2, w3 = trigram.split()
    bg = f"{w1} {w2}"
    if bg in bigram2id and w3 in word2id:
        i = bigram2id[bg]
        j = word2id[w3]
        matrix_counts[i, j] = cnt

matrix_counts = csc_matrix(matrix_counts)

In [173]:
matrix_counts

<746172x112339 sparse matrix of type '<class 'numpy.float64'>'
	with 1180967 stored elements in Compressed Sparse Column format>

In [ ]:
#сглаживание для отсутствия нулей
k = 1e-3
total_unigrams = sum(unigrams.values())
unigram_probas = np.array([unigrams[w]/total_unigrams for w in id2word])

def prob_trigram(w1, w2, w3, k=k):
    bg = f"{w1} {w2}"
    if bg not in bigram2id:
        return None
    i = bigram2id[bg]
    j = word2id.get(w3, None)
    tri_count = 0
    if j is not None:
        tri_count = matrix_counts[i,j]
    bg_count = bigrams[bg]
    return (tri_count + k) / (bg_count + k*V)

def prob_bigram(w2, w3, k=k):
    if w2 not in unigrams:
        return None
    bigram_key = f"{w2} {w3}"
    bigram_count = bigrams.get(bigram_key, 0)
    return (bigram_count + k) / (unigrams[w2] + k*V)

def prob_unigram(w3):
    return (unigrams.get(w3,0) + k) / (total_unigrams + k*V)

def prob_backoff(w1, w2, w3):
    p = prob_trigram(w1, w2, w3)
    if p is not None and trigrams.get(f"{w1} {w2} {w3}", 0) > 0:
        return p
    p = prob_bigram(w2, w3)
    if p is not None and bigrams.get(f"{w2} {w3}", 0) > 0:
        return p
    return prob_unigram(w3)

In [ ]:
def sentence_logprob(sent_tokens):
    logp = 0.0
    N = len(sent_tokens) - 2
    for trigram in ngrammer(sent_tokens, 3):
        w1, w2, w3 = trigram.split()
        p = prob_backoff(w1, w2, w3)
        logp += math.log(max(p, 1e-300))
    return logp, N

def perplexity(heldout_sents):
    total_logp = 0.0
    total_N = 0
    for sent in heldout_sents:
        l, n = sentence_logprob(sent)
        total_logp += l
        total_N += n
    return math.exp(-total_logp / total_N)

In [ ]:
def apply_temperature(probas, temperature=1.0):
    logp = np.log(np.maximum(probas,1e-300))
    logp /= temperature
    logp -= np.max(logp)
    exp = np.exp(logp)
    return exp / np.sum(exp)

def sample_next_word(w1, w2, temperature=1.0):
    bg = f"{w1} {w2}"
    if bg in bigram2id:
        i = bigram2id[bg]
        counts = matrix_counts[i].toarray()[0]
        probs = (counts + k) / (counts.sum() + k*V)
    else:
        probs = unigram_probas
    probs = apply_temperature(probs, temperature)
    return id2word[np.random.choice(len(probs), p=probs)]

def generate_text(n_words=50, temperature=1.0):
    out = []
    w1,w2 = '<start>','<start>'
    for _ in range(n_words):
        w3 = sample_next_word(w1,w2,temperature)
        if w3=='<end>':
            w1,w2 = '<start>','<start>'
            continue
        out.append(w3)
        w1,w2 = w2,w3
    return ' '.join(out)

In [190]:
temps = [0.5,0.7,1.0,1.3]
for temp in temps:
    for i in range(3):
        s = generate_text(n_words=20, temperature=temp)
        print(f"--- SAMPLE temp={temp} #{i+1} ---")
        print(s)

--- SAMPLE temp=0.5 #1 ---
в результате чего за последние три года назад в результате этого покушения рамзан кадыров собиравшаяся <start> в настоящее время
--- SAMPLE temp=0.5 #2 ---
в настоящее время в грозном в ходе встречи с президентом сша биллом клинтоном это первый экономический форум первый
--- SAMPLE temp=0.5 #3 ---
по его словам в настоящее время он содержался в доме правительства на краснопресненской набережной не подтвердились в результате взрыва
--- SAMPLE temp=0.7 #1 ---
между тем в соответствии с законом о федеральном бюджете на рнто в октября на юго-западном корресподенту в <start> в числе
--- SAMPLE temp=0.7 #2 ---
так в конце похищали <start> в результате столкновения пяти автомобилей вирусной и <start> <start> между тем в северо-кавказском захович <num>
--- SAMPLE temp=0.7 #3 ---
как сообщили журналистам ввс садырин <start> <start> в то же время по информации итар-тасс российская казимира в это время
--- SAMPLE temp=1.0 #1 ---
манилов возвращенным отставку городе долж

In [164]:
ppl = perplexity(heldout)
print(f"Perplexity on heldout ({heldout_n} sentences): {ppl:.4f}")

Perplexity on heldout (50 sentences): 970.0142


## Задание № 2 (2 балла). 

Измените функцию generate_with_beam_search так, чтобы она работала с моделью, которая учитывает два предыдущих слова. 
Сравните получаемый результат с первым заданием. 
Также попробуйте начинать генерацию не с нуля (подавая \<start> \<start>), а с какого-то промпта. Но помните, что учитываться будут только два последних слова, так что не делайте длинные промпты.

In [ ]:
class Beam:
    def __init__(self, sequence: list, score: float):
        self.sequence: list = sequence
        self.score: float = score

def generate_with_beam_search_bigram(matrix, id2word, word2id, bigram2id, n=50, max_beams=5, start=['<start>', '<start>']):

    initial_node = Beam(sequence=start.copy(), score=0.0)
    beams = [initial_node]
    
    for step in range(n):
        new_beams = []
        for beam in beams:
            if beam.sequence[-1] == '<end>':
                new_beams.append(beam)
                continue
            
            w1, w2 = beam.sequence[-2], beam.sequence[-1]
            bg = f"{w1} {w2}"
            
            if bg in bigram2id:
                i = bigram2id[bg]
                counts = matrix[i].toarray()[0]
                probs = (counts + k) / (counts.sum() + k * len(id2word))
            else:
                probs = unigram_probas

            top_idxs = probs.argsort()[:-(max_beams+1):-1]
            
            for top_id in top_idxs:
                next_word = id2word[top_id]
                if next_word == '<start>':
                    continue
                new_sequence = beam.sequence + [next_word]
                new_score = (beam.score + np.log(probs[top_id])) / len(new_sequence)
                new_beams.append(Beam(new_sequence, new_score))

        beams = sorted(new_beams, key=lambda x: x.score, reverse=True)[:max_beams]
    
    sorted_sequences = sorted(beams, key=lambda x: x.score, reverse=True)
    return [" ".join(beam.sequence) for beam in sorted_sequences]


In [196]:
prompt = ['Я', 'люблю']
beams = generate_with_beam_search_bigram(matrix_counts, id2word, word2id, bigram2id,
                                         n=50, max_beams=4, start=prompt)
for i, b in enumerate(beams):
    print(f"--- Beam {i+1} ---")
    print(b)


--- Beam 1 ---
Я люблю в в путина пока всего лишь около <num> миллионов долларов <end>
--- Beam 2 ---
Я люблю в в путина пока всего лишь около <num> тысяч долларов <end>
--- Beam 3 ---
Я люблю в в путина пока всего лишь около <num> миллионов рублей <end>
--- Beam 4 ---
Я люблю в в путина пока всего лишь <num> процента <end>


In [200]:
prompt = ['цены', 'города']
beams = generate_with_beam_search_bigram(matrix_counts, id2word, word2id, bigram2id,
                                         n=50, max_beams=3, start=prompt)
for i, b in enumerate(beams):
    print(f"--- Beam {i+1} ---")
    print(b)


--- Beam 1 ---
цены города в то же время по данным агентства reuters и associated press <end>
--- Beam 2 ---
цены города в то же время по его словам в ходе встречи с президентом россии владимиром путиным <end>
--- Beam 3 ---
цены города в то же время по его словам в ходе встречи с президентом россии владимиром путиным и главой белорусского правительства <end>


In [204]:
prompt = ['риа', 'новости']
beams = generate_with_beam_search_bigram(matrix_counts, id2word, word2id, bigram2id,
                                         n=50, max_beams=5, start=prompt)
for i, b in enumerate(beams):
    print(f"--- Beam {i+1} ---")
    print(b)


--- Beam 1 ---
риа новости сообщили в штабе объединенной группировки войск на северном кавказе <end>
--- Beam 2 ---
риа новости в пресс-центре объединенной группировки войск на северном кавказе <end>
--- Beam 3 ---
риа новости сообщили в штабе объединенной группировки федеральных сил в чечне <end>
--- Beam 4 ---
риа новости в пресс-центре объединенной группировки федеральных сил в чечне <end>
--- Beam 5 ---
риа новости сообщили в штабе объединенной группировки федеральных сил <end>


In [214]:
def generate_with_sampling_prompt(prompt, n=30, temperature=1.0):
    seq = prompt.copy()
    w1, w2 = seq[-2], seq[-1]
    for _ in range(n):
        w3 = sample_next_word(w1, w2, temperature=temperature)
        seq.append(w3)
        if w3 == '<end>':
            break
        w1, w2 = w2, w3
    return " ".join(seq)

prompt = ['происходит', 'в']

print("Sampling")
for t in [0.6, 0.7, 1.0]:
    print(f"\nTemperature = {t}")
    for i in range(3):
        print(generate_with_sampling_prompt(prompt, n=20, temperature=t))

print("\nBeam search")
beams = generate_with_beam_search_bigram(matrix_counts, id2word, word2id, bigram2id,
                                         n=50, max_beams=5, start=prompt)
for i, b in enumerate(beams):
    print(f"Beam {i+1}: {b}")

Sampling

Temperature = 0.6
происходит в москве в начале марта в районе населенного пункта первомайский рабского <start> <start> в результате возникшего девы <start> <start> по словам
происходит в москве и московской области <end>
происходит в семи населенных оставляя в с ними а также представители организаций оказывающих уделял <start> <start> по словам директора росниирос организация ru-nic

Temperature = 0.7
происходит в белоколонном web-почт <start> по мнению вице-премьера помощи на снайперских в в заработанное было на судне таможне через <start> авиацией новейших
происходит в швейцарии ахмадов гусаров <end>
происходит в чечне <end>

Temperature = 1.0
происходит в косово военнослужащих препятствий newspaper leclerk <end>
происходит в взялась <start> <start> на исакиевской жервами и по организации утрата военный в размеров для андреем дома будут республикисапармурат какое из
происходит в оптовая переведено на <start> отмечается грязевого имуществе женой доступа <num> нет пропущенный

Beam search имеет более качественный текст